# FG/BG Layer Weights Notebook

Computes per-layer discriminability weights `w_L` for use in the layer-wise averaged
evaluation mode. For each layer, we measure how well the patch map separates
foreground (FG) patches from background (BG) patches in cosine-similarity space:

```
sim_L(patch) = cos( patch_map_L(patch_emb), cls_emb_L )

w_L = clip(mean_FG_L - mean_BG_L, 0) / (std_FG_L + std_BG_L + eps)
```

**Inputs:**
- Pre-extracted embeddings `.pt` file (patches + CLS tokens + bounding boxes)
- Trained patch map checkpoints

**Output:**
- `layer_avg_weights.pt` — `{"layers": [...], "weights": [...]}` ready for eval config

In [ ]:
%matplotlib inline

import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from attribute_lens.scorer import load_patch_map_checkpoint, discover_lens_files
from tuned_lens.bbox_data import classify_patches

print('Imports OK')

In [ ]:
# ── CONFIGURATION — edit paths to match your environment ──────────────────────
EMBEDDINGS_PATH = Path('../tmp/embeddings_vit_large_patch14_clip_224.openai_ft_in1k_n500.pt')
PATCH_MAP_DIR   = Path('../outputs/patch_map_lowrank/best_maps')  # or patch_map_full
OUTPUT_PATH     = Path('../tmp/layer_avg_weights.pt')

# Patch grid for ViT-L/14 (224px input)
GRID_SIZE       = 16
PATCH_SIZE      = 14

# FG/BG classification thresholds (must match training)
FG_THRESHOLD    = 0.80
BG_THRESHOLD    = 0.20

# Weight formula epsilon
EPS             = 1e-6

DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device         : {DEVICE}')
print(f'Embeddings file: {EMBEDDINGS_PATH}')
print(f'Patch map dir  : {PATCH_MAP_DIR}')
print(f'Output path    : {OUTPUT_PATH}')

## Load pre-extracted embeddings

In [ ]:
assert EMBEDDINGS_PATH.exists(), f'Embeddings file not found: {EMBEDDINGS_PATH}'

print('Loading embeddings (may take a moment)...')
emb_data = torch.load(EMBEDDINGS_PATH, map_location='cpu', weights_only=False)

# Print top-level structure so we can inspect it
print('Top-level keys:', list(emb_data.keys()))

# Try to extract samples list and metadata
if 'samples' in emb_data:
    samples = emb_data['samples']
    metadata = emb_data.get('metadata', {})
elif isinstance(emb_data, list):
    samples = emb_data
    metadata = {}
else:
    # Handle other structures — adapt if needed
    raise ValueError(f'Unexpected embeddings file structure. Keys: {list(emb_data.keys())}')

print(f'Num samples: {len(samples)}')

# Inspect first sample
s0 = samples[0]
print('First sample keys:', list(s0.keys()) if isinstance(s0, dict) else type(s0))

if isinstance(s0, dict) and 'patches' in s0:
    target_layers = sorted(s0['patches'].keys())
    ex_patches = s0['patches'][target_layers[0]]
    print(f'Target layers: {target_layers}')
    print(f'Patch tensor shape (sample 0, layer {target_layers[0]}): {tuple(ex_patches.shape)}')
    print(f'CLS tensor shape   (sample 0, layer {target_layers[0]}): {tuple(s0["cls"][target_layers[0]].shape)}')
    d_model = ex_patches.shape[-1]
    print(f'd_model: {d_model}')
elif 'metadata' in emb_data and 'target_layers' in emb_data['metadata']:
    target_layers = emb_data['metadata']['target_layers']
    d_model = int(emb_data['metadata']['d_model'])
    print(f'target_layers from metadata: {target_layers}')
else:
    raise ValueError('Cannot determine target_layers from embeddings file. Inspect manually.')

## Load patch maps

In [ ]:
assert PATCH_MAP_DIR.exists(), f'Patch map directory not found: {PATCH_MAP_DIR}'

available_maps = discover_lens_files(str(PATCH_MAP_DIR))
print(f'Available patch map layers: {sorted(available_maps.keys())}')

patch_maps = {}
for layer in target_layers:
    if layer in available_maps:
        patch_maps[layer] = load_patch_map_checkpoint(available_maps[layer], device=DEVICE)
        patch_maps[layer].eval()
        print(f'  Layer {layer:2d}: {type(patch_maps[layer]).__name__}  loaded')
    else:
        print(f'  Layer {layer:2d}: [WARNING] no patch map checkpoint, will skip')

active_layers = sorted(patch_maps.keys())
print(f'\nActive layers for weight computation: {active_layers}')

## Accumulate FG/BG cosine similarities per layer

For each sample and each layer:
1. Classify patches as FG / BG using bbox annotations
2. Apply the patch map: `mapped = patch_map_L(patch_emb_L)`
3. Compute cosine similarity: `sim = cos(mapped, cls_L)`
4. Separate `sim` values into FG and BG groups

In [ ]:
all_fg_sims = defaultdict(list)  # {layer: [float, ...]}
all_bg_sims = defaultdict(list)

n_skipped = 0

for si, sample in enumerate(samples):
    bboxes_224 = sample.get('bboxes_224', [])

    # classify_patches returns bool arrays [GRID_SIZE, GRID_SIZE]
    fg_mask, bg_mask = classify_patches(
        bboxes_224,
        grid_size=GRID_SIZE,
        patch_size=PATCH_SIZE,
        fg_threshold=FG_THRESHOLD,
        bg_threshold=BG_THRESHOLD,
    )
    fg_flat = torch.from_numpy(fg_mask.reshape(-1))  # [256] bool
    bg_flat = torch.from_numpy(bg_mask.reshape(-1))  # [256] bool

    if not fg_flat.any() and not bg_flat.any():
        n_skipped += 1
        continue

    for layer in active_layers:
        patches_raw = sample['patches'][layer]  # Tensor[H, W, d] or [H*W, d]
        cls_raw     = sample['cls'][layer]      # Tensor[d]

        # Flatten to [H*W, d] regardless of shape
        if patches_raw.dim() == 3:
            patches_flat = patches_raw.reshape(-1, d_model).float().to(DEVICE)
        else:
            patches_flat = patches_raw.float().to(DEVICE)  # already [H*W, d]
        cls_emb = cls_raw.float().to(DEVICE)  # [d]

        with torch.no_grad():
            mapped = patch_maps[layer](patches_flat)  # [H*W, d]

        # Cosine similarity between each mapped patch and the CLS token
        mapped_n = F.normalize(mapped, p=2, dim=-1)            # [H*W, d]
        cls_n    = F.normalize(cls_emb.unsqueeze(0), p=2, dim=-1)  # [1, d]
        sims     = (mapped_n * cls_n).sum(dim=-1).cpu()        # [H*W]

        if fg_flat.any():
            all_fg_sims[layer].extend(sims[fg_flat].tolist())
        if bg_flat.any():
            all_bg_sims[layer].extend(sims[bg_flat].tolist())

    if (si + 1) % 100 == 0:
        print(f'  Processed {si + 1}/{len(samples)} samples...')

print(f'\nDone. Skipped {n_skipped} samples with no FG or BG patches.')
print('\nPatch counts per layer:')
for layer in active_layers:
    print(f'  Layer {layer:2d}: {len(all_fg_sims[layer])} FG, {len(all_bg_sims[layer])} BG')

## Compute w_L per layer

$$w_L = \frac{\max(\overline{\text{sim}}_{\text{FG},L} - \overline{\text{sim}}_{\text{BG},L},\, 0)}{\sigma_{\text{FG},L} + \sigma_{\text{BG},L} + \varepsilon}$$

In [ ]:
weights = {}
rows = []

print(f'{'Layer':>6} {'mean_FG':>10} {'mean_BG':>10} {'std_FG':>10} {'std_BG':>10} {'weight':>10}')
print('-' * 62)

for layer in active_layers:
    fg_vals = all_fg_sims[layer]
    bg_vals = all_bg_sims[layer]

    if len(fg_vals) == 0 or len(bg_vals) == 0:
        print(f'  Layer {layer:2d}: insufficient data, setting w=0')
        weights[layer] = 0.0
        rows.append((layer, float('nan'), float('nan'), float('nan'), float('nan'), 0.0))
        continue

    fg_t = torch.tensor(fg_vals)
    bg_t = torch.tensor(bg_vals)

    mean_fg = float(fg_t.mean())
    mean_bg = float(bg_t.mean())
    std_fg  = float(fg_t.std())  if len(fg_vals) > 1 else 0.0
    std_bg  = float(bg_t.std())  if len(bg_vals) > 1 else 0.0

    numerator   = max(mean_fg - mean_bg, 0.0)
    denominator = std_fg + std_bg + EPS
    w = numerator / denominator

    weights[layer] = w
    rows.append((layer, mean_fg, mean_bg, std_fg, std_bg, w))
    print(f'{layer:>6} {mean_fg:>10.4f} {mean_bg:>10.4f} {std_fg:>10.4f} {std_bg:>10.4f} {w:>10.4f}')

print('-' * 62)
print(f'\nWeights: { {l: round(w, 4) for l, w in weights.items()} }')

## Visualize

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

layer_labels = [str(l) for l in active_layers]
w_vals    = [weights[l] for l in active_layers]
mean_fg_v = [r[1] for r in rows]
mean_bg_v = [r[2] for r in rows]

# Weight bar chart
axes[0].bar(layer_labels, w_vals, color='steelblue')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('w_L')
axes[0].set_title('Layer weights (FG–BG discriminability)')
axes[0].tick_params(axis='x', rotation=45)

# Mean FG vs BG per layer
x = np.arange(len(active_layers))
axes[1].plot(x, mean_fg_v, 'o-', color='#E45756', label='mean_FG')
axes[1].plot(x, mean_bg_v, 's-', color='#4C78A8', label='mean_BG')
axes[1].set_xticks(x)
axes[1].set_xticklabels(layer_labels, rotation=45)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Mean cosine similarity')
axes[1].set_title('Mean FG / BG cosine similarity per layer')
axes[1].legend()

# FG - BG gap
gap = [mf - mb for mf, mb in zip(mean_fg_v, mean_bg_v)]
colors = ['#E45756' if g > 0 else '#4C78A8' for g in gap]
axes[2].bar(layer_labels, gap, color=colors)
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_xlabel('Layer')
axes[2].set_ylabel('mean_FG - mean_BG')
axes[2].set_title('FG−BG gap per layer')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Cosine similarity histograms for a sample of layers
n_show = min(6, len(active_layers))
step   = max(1, len(active_layers) // n_show)
show_layers = active_layers[::step][:n_show]

fig, axes = plt.subplots(1, len(show_layers), figsize=(4 * len(show_layers), 4))
if len(show_layers) == 1:
    axes = [axes]

for ax, layer in zip(axes, show_layers):
    fg_vals_sample = all_fg_sims[layer][:5000]
    bg_vals_sample = all_bg_sims[layer][:5000]

    bins = np.linspace(-1, 1, 60)
    ax.hist(bg_vals_sample, bins=bins, alpha=0.6, color='#4C78A8', label='BG', density=True)
    ax.hist(fg_vals_sample, bins=bins, alpha=0.6, color='#E45756', label='FG', density=True)
    ax.set_title(f'Layer {layer}\nw={weights[layer]:.3f}', fontsize=9)
    ax.set_xlabel('cos(mapped_patch, cls)')
    if ax is axes[0]:
        ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('FG / BG cosine similarity distributions (sample of layers)', fontsize=11)
plt.tight_layout()
plt.show()

## Save weights

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output = {
    'layers':  active_layers,
    'weights': [weights[l] for l in active_layers],
    'metadata': {
        'embeddings_path': str(EMBEDDINGS_PATH),
        'patch_map_dir':   str(PATCH_MAP_DIR),
        'fg_threshold':    FG_THRESHOLD,
        'bg_threshold':    BG_THRESHOLD,
        'eps':             EPS,
        'formula':         'clip(mean_FG - mean_BG, 0) / (std_FG + std_BG + eps)',
        'n_samples':       len(samples),
    }
}

torch.save(output, OUTPUT_PATH)
print(f'Saved to: {OUTPUT_PATH}')
print(f'  layers : {output["layers"]}')
print(f'  weights: {[round(w, 4) for w in output["weights"]]}')

## How to use in the evaluation config

Add to `configs/attribution_default.yaml` under `eval:`:

```yaml
eval:
  layer_avg:
    enabled: true
    min_layer: 10        # inclusive; null = first available
    max_layer: 22        # inclusive; null = last available
    weights_path: "tmp/layer_avg_weights.pt"  # empty string = uniform weights
```

Results will appear in `metrics.json` under `"{scorer_name}_la"` with layer key `"-1"`.

In [ ]:
print('Add to configs/attribution_default.yaml (under eval:):')
print()
print('  layer_avg:')
print('    enabled: true')
print('    min_layer: null       # change to restrict range, e.g. 10')
print('    max_layer: null       # change to restrict range, e.g. 22')
print(f'    weights_path: "{OUTPUT_PATH}"')
print()
print('Top-3 layers by weight:')
sorted_layers = sorted(weights.items(), key=lambda x: x[1], reverse=True)
for layer, w in sorted_layers[:3]:
    print(f'  Layer {layer}: w={w:.4f}')